In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive')

In [ ]:
os.listdir('/content/drive/MyDrive/Datasets')

In [ ]:
file_path = '/content/drive/MyDrive/Datasets/indian-job-market-dataset-2025.xlsx'
df = pd.read_excel(file_path)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df_clean = df.dropna(subset=['tagsAndSkills']).copy()
df_clean['skills_clean'] = df_clean['tagsAndSkills'].str.lower()
df_clean['skills_clean'] = df_clean['skills_clean'].apply(
    lambda x: re.sub(r'[^a-zA-Z, ]', '', x)
)
df_clean['skills_list'] = df_clean['skills_clean'].str.split(',')
all_skills = [skill.strip() for sublist in df_clean['skills_list'] for skill in sublist if skill.strip() != ""]
skill_counts = Counter(all_skills)
skills_df = pd.DataFrame(skill_counts.items(), columns=['Skill', 'Count'])
skills_df = skills_df.sort_values(by='Count', ascending=False)
skills_df.head(20)

In [ ]:
skill_mapping = {
    'ms excel': 'excel',
    'microsoft excel': 'excel',
    'excel ': 'excel',
    'powerbi': 'power bi',
    'power bi ': 'power bi',
    'communication skills': 'communication',
    'english': 'communication',
    'c++': 'c',
    'js': 'javascript',
    'customer service': 'customer support',
    'sales ': 'sales'
}
def clean_skill(skill):
    skill = skill.strip()
    return skill_mapping.get(skill, skill)
skills_df['Skill'] = skills_df['Skill'].apply(clean_skill)
skills_df = skills_df.groupby('Skill', as_index=False)['Count'].sum()
skills_df = skills_df.sort_values(by='Count', ascending=False)
remove_skills = [
    'management', 'development', 'communication',
    'troubleshooting', 'customer support'
]
skills_df = skills_df[~skills_df['Skill'].isin(remove_skills)]
skills_df.head(20)

In [ ]:
tech_skills = [
    'python', 'java', 'c', 'javascript', 'sql', 'css'
]
business_skills = [
    'sales', 'marketing', 'project management', 'business development', 'accounting'
]
tools_skills = [
    'excel', 'power bi', 'sap', 'automation'
]
def categorize(skill):
    if skill in tech_skills:
        return 'Tech'
    elif skill in business_skills:
        return 'Business'
    elif skill in tools_skills:
        return 'Tools'
    else:
        return 'Other'
skills_df['Category'] = skills_df['Skill'].apply(categorize)
category_df = skills_df.groupby('Category')['Count'].sum().reset_index()
category_df = category_df.sort_values(by='Count', ascending=False)
category_df

In [ ]:
tech_skills = [
    'python', 'java', 'c', 'javascript', 'sql', 'css', 'html',
    'machine learning', 'data analysis', 'data science', 'deep learning',
    'artificial intelligence', 'aws', 'cloud', 'react', 'nodejs'
]
business_skills = [
    'sales', 'marketing', 'project management', 'business development',
    'accounting', 'finance', 'operations', 'strategy', 'consulting'
]
tools_skills = [
    'excel', 'power bi', 'sap', 'tableau', 'automation',
    'google analytics', 'crm', 'erp'
]
def categorize(skill):
    skill = skill.lower()
    if any(ts in skill for ts in tech_skills):
        return 'Tech'
    elif any(bs in skill for bs in business_skills):
        return 'Business'
    elif any(tl in skill for tl in tools_skills):
        return 'Tools'
    else:
        return 'Other'
skills_df['Category'] = skills_df['Skill'].apply(categorize)
category_df = skills_df.groupby('Category')['Count'].sum().reset_index()
category_df = category_df.sort_values(by='Count', ascending=False)
category_df

In [ ]:
top_skills_by_category = {}
for category in skills_df['Category'].unique():
    top_skills = skills_df[skills_df['Category'] == category] \
                    .sort_values(by='Count', ascending=False) \
                    .head(10)
    top_skills_by_category[category] = top_skills[['Skill', 'Count']]
for category, data in top_skills_by_category.items():
    print(f"\nTop skills in {category}:\n")
    print(data)

In [ ]:
data_roles = df_clean[
    df_clean['title'].str.lower().str.contains(
        'data analyst|data scientist|data engineer|analytics', regex=True
    )
]
print("Total data-related jobs:", data_roles.shape[0])
skills_series = data_roles['skills_clean'].str.split(',')
all_skills_data = [
    skill.strip()
    for sublist in skills_series
    for skill in sublist
    if skill.strip() != ""
]
from collections import Counter
data_skill_counts = Counter(all_skills_data)
data_skills_df = pd.DataFrame(data_skill_counts.items(), columns=['Skill', 'Count'])
data_skills_df = data_skills_df.sort_values(by='Count', ascending=False)
data_skills_df.head(15)

In [ ]:
top_overall = skills_df.head(20)
top_data = data_skills_df.head(20)
overall_skills_set = set(top_overall['Skill'])
data_skills_set = set(top_data['Skill'])
common_skills = overall_skills_set.intersection(data_skills_set)
data_unique = data_skills_set - overall_skills_set
overall_unique = overall_skills_set - data_skills_set
print("Common Skills:\n", common_skills)
print("\nData Role Exclusive Skills:\n", data_unique)
print("\nGeneral Market Skills (Not in Data Roles):\n", overall_unique)

In [ ]:
def categorize_advanced(skill):
    skill = skill.lower()
    if any(x in skill for x in [
        'python','java','c','c++','javascript','sql','html','css',
        'react','node','angular'
    ]):
        return 'Programming'
    elif any(x in skill for x in [
        'machine learning','data','analytics','ai','artificial intelligence',
        'deep learning','nlp','computer vision'
    ]):
        return 'Data & AI'
    elif any(x in skill for x in [
        'aws','azure','gcp','hadoop','spark','hive','pyspark',
        'airflow','etl','kubernetes','docker'
    ]):
        return 'Cloud & Big Data'
    elif any(x in skill for x in [
        'sales','marketing','business','operations','strategy',
        'consulting','project management','accounting','finance'
    ]):
        return 'Business & Management'
    elif any(x in skill for x in [
        'excel','power bi','tableau','sap','erp','crm','oracle'
    ]):
        return 'Tools & Enterprise'
    elif any(x in skill for x in [
        'communication','team','leadership','interpersonal',
        'presentation','negotiation'
    ]):
        return 'Soft Skills'
    else:
        return 'Other'
skills_df['Better_Category'] = skills_df['Skill'].apply(categorize_advanced)
better_category_df = skills_df.groupby('Better_Category')['Count'].sum().reset_index()
better_category_df = better_category_df.sort_values(by='Count', ascending=False)
better_category_df

In [ ]:
!pip install plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
top_overall = skills_df.head(8)
top_data = data_skills_df.head(8)
gap_counts = [len(common_skills), len(data_unique), len(overall_unique)]
gap_labels = ['Common', 'Data Only', 'General Only']
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Top Skills (Overall)",
        "Category Distribution",
        "Top Skills (Data Roles)",
        "Skill Gap Overview"
    ),
    specs=[[{"type": "bar"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]],
    horizontal_spacing=0.12,
    vertical_spacing=0.18
)
fig.add_trace(
    go.Bar(
        x=top_overall['Count'],
        y=top_overall['Skill'],
        orientation='h',
        text=top_overall['Count'],
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='
    ),
    row=1, col=1
)
fig.add_trace(
    go.Pie(
        labels=better_category_df['Better_Category'],
        values=better_category_df['Count'],
        textinfo='percent',
        textfont=dict(color='white')
    ),
    row=1, col=2
)
fig.add_trace(
    go.Bar(
        x=top_data['Count'],
        y=top_data['Skill'],
        orientation='h',
        text=top_data['Count'],
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='
    ),
    row=2, col=1
)
fig.add_trace(
    go.Bar(
        x=gap_labels,
        y=gap_counts,
        text=gap_counts,
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='
    ),
    row=2, col=2
)
fig.update_layout(
    height=650,
    width=1200,
    title={
        'text': "Indian Job Market Skill Gap Dashboard",
        'x': 0.5,
        'font': {'size': 24, 'color': 'white'}
    },
    template="plotly_dark",
    showlegend=False,
    margin=dict(l=60, r=60, t=100, b=50),
    font=dict(color="white")
)
fig.update_annotations(font_size=14)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=2, col=1)
fig.show()
file_name = "skill_gap_dashboard.html"
html_string = f"""
<html>
<head>
    <style>
        body {{
            margin: 0;
            background-color:
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }}
        .dashboard {{
            padding: 20px;
            border-radius: 12px;
            background-color:
            box-shadow: 0 0 25px rgba(0,0,0,0.8);
            border: 1px solid
        }}
    </style>
</head>
<body>
    <div class="dashboard">
        <div id="chart"></div>
    </div>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <script>
        var fig = {fig.to_json()};
        Plotly.newPlot('chart', fig.data, fig.layout);
    </script>
</body>
</html>
"""
with open(file_name, "w") as f:
    f.write(html_string)
files.download(file_name)